In [ ]:
!pip install scikit-learn
!pip install transformers
!pip install datasets
!pip install torch

In [2]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 89.6 gigabytes of available RAM

You are using a high-RAM runtime!


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from datasets import Dataset


In [3]:
import random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic behavior (important for papers)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print('No GPU detected, using CPU instead')

GPU: NVIDIA A100-SXM4-40GB


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Load real and synthetic datasets
real_df = pd.read_csv('/content/drive/MyDrive/Honour/DatasetV4.csv', dtype={"ActionabilityLevel": int})
synthetic_df = pd.read_csv('/content/drive/MyDrive/Honour/synthCS.csv', dtype={"ActionabilityLevel": int})

print('Number of real data rows: {:,}'.format(real_df.shape[0]))
print('Number of synthetic data rows: {:,}\n'.format(synthetic_df.shape[0]))

Number of real data rows: 1,887
Number of synthetic data rows: 1,187



In [7]:
counts = real_df['ActionabilityLevel'].value_counts().sort_index()
print(counts)

counts = synthetic_df['ActionabilityLevel'].value_counts().sort_index()
print(counts)

ActionabilityLevel
0    967
1    681
2    239
Name: count, dtype: int64
ActionabilityLevel
0    211
1    348
2    628
Name: count, dtype: int64


In [180]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bclavie/edubert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

MAX_LEN = 256

tokenizer_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/520 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [181]:
class FeedbackDataset(Dataset):
    def __init__(self, df, tokenizer, MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len = MAX_LEN

        # Apply textpreprocessing to all feedback
        self.feedback = [str(fb) for fb in df['Feedback'].tolist()]
        self.labels = df['ActionabilityLevel'].tolist()

    def __len__(self):
        return len(self.feedback)

    def __getitem__(self, idx):
        feedbacks = [self.feedback[i] for i in idx]
        labels = [self.labels[i] for i in idx]
        encodings = self.tokenizer(
            feedbacks,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encodings['input_ids'],
            'attention_mask': encodings['attention_mask'],
            'labels': torch.tensor(labels, dtype=torch.long)
        }

In [182]:
# Count synthetic vs real in each split
def count_source(df_name, df):
    n_real = df['Source'].eq('real').sum() if 'Source' in df.columns else 0
    n_synth = df['Source'].eq('synthetic').sum() if 'Source' in df.columns else 0
    total = len(df)
    print(f"{df_name} set:")
    print(f"Total samples: {total}")
    print(f"Real: {n_real}")
    print(f"Synthetic: {n_synth}\n")

# Label your dataframes before combining
real_df['Source'] = 'real'
synthetic_df['Source'] = 'synthetic'

# Build train/val/test
train_real, temp_real = train_test_split(real_df, test_size=0.5, random_state=46)
val_real, test_real = train_test_split(temp_real, test_size=0.2, random_state=43)

train_df = pd.concat([train_real, synthetic_df]).sample(frac=1, random_state=42).reset_index(drop=True)

val_df = val_real
test_df = test_real

# Show counts by source
for name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    count_source(name, df)

# Show Actionability class distribution per split
def class_distribution(df_name, df):
    print(f"{df_name} Class distribution:")
    print(df['ActionabilityLevel'].value_counts().sort_index())
    print()

for name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    class_distribution(name, df)

# Build datasets
training_set = FeedbackDataset(train_df, tokenizer, MAX_LEN)
testing_set = FeedbackDataset(test_df, tokenizer, MAX_LEN)
val_set = FeedbackDataset(val_df, tokenizer, MAX_LEN)


Train set:
Total samples: 2130
Real: 943
Synthetic: 1187

Validation set:
Total samples: 755
Real: 755
Synthetic: 0

Test set:
Total samples: 189
Real: 189
Synthetic: 0

Train Class distribution:
ActionabilityLevel
0    667
1    701
2    762
Name: count, dtype: int64

Validation Class distribution:
ActionabilityLevel
0    414
1    254
2     87
Name: count, dtype: int64

Test Class distribution:
ActionabilityLevel
0    97
1    74
2    18
Name: count, dtype: int64



In [183]:
import os
save_dir = "/content/drive/MyDrive/Honour/Finetuning"
os.makedirs(save_dir, exist_ok=True)

train_df.to_csv(os.path.join(save_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(save_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(save_dir, "test.csv"), index=False)

In [184]:
example = training_set[[2]]

input_ids = example['input_ids'][0]
attention_mask = example['attention_mask'][0]
label = example['labels'][0].item()

tokens = tokenizer.convert_ids_to_tokens(input_ids)
decoded_text = tokenizer.decode(input_ids, skip_special_tokens=True)

print(f"Label: {label}\n")
print(f"Token IDs: {input_ids.tolist()}\n")
print(f"Tokens: {tokens}\n")
print(f"Decoded text: {decoded_text}")


Label: 2

Token IDs: [101, 5136, 25416, 18908, 28741, 2290, 1996, 9089, 2098, 15932, 1999, 1996, 2832, 2850, 2696, 3853, 2000, 5335, 2836, 1998, 3191, 8010, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Tokens: ['[CLS]', 'consider', 'ref', '##act', '##orin', '##g', 'the', 'nest', '##ed', 'loops', 'in', 'the', 'process', '##da', '##ta', 

In [185]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch.nn as nn

In [186]:
import torch
import torch.nn as nn

class BertClassifier(nn.Module):
    def __init__(self, num_labels=3, dropout_prob=0.1):
        super(BertClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained("bclavie/edubert")
        hidden_size = self.bert.config.hidden_size

        self.pre_classifier = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state
        pooled_output = hidden_state[:, 0]

        x = self.pre_classifier(pooled_output)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits


In [208]:
from google.colab import drive
drive.mount('/content/drive')

SEED = 1
set_seed(SEED)
model = BertClassifier(num_labels=3)
model.to(device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [209]:
EPOCHS = 12
BATCH_SIZE = 12
LEARNING_RATE = 2e-5
GRADIENT_ACCUMULATION_STEPS = 2

In [210]:
from torch.utils.data import DataLoader
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    training_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=g
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    testing_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


In [211]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
total_steps = len(train_loader) * EPOCHS // GRADIENT_ACCUMULATION_STEPS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

import torch.nn.functional as F
loss_fn = nn.CrossEntropyLoss()

scaler = torch.amp.GradScaler("cuda")


In [212]:
from tqdm import tqdm
import torch

def train_epoch(model, data_loader, loss_fn, optimizer, scheduler, device, scaler):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch_idx, batch in enumerate(tqdm(data_loader), start=1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # optimizer.zero_grad()
        #with torch.amp.autocast(device_type=device.type if device.type != 'mps' else 'cpu'):
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        if batch_idx % GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        preds = logits.argmax(dim=1)
        correct = (preds == labels).sum().item()
        total_loss += loss.item() * labels.size(0)
        total_correct += correct
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

def eval_model(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)

            batch_size = labels.size(0)
            preds = logits.argmax(dim=1)
            correct = (preds == labels).sum().item()

            total_loss += loss.item() * batch_size
            total_correct += correct
            total_samples += batch_size

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc, all_preds, all_labels


best_val_acc = 0.0
for epoch in range(EPOCHS):
    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")
    train_loss, train_acc = train_epoch(
        model, train_loader, loss_fn, optimizer, scheduler, device, scaler
    )
    print(f"Train → Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")

    val_loss, val_acc, val_preds, val_labels = eval_model(
        model, val_loader, loss_fn, device
    )
    print(f"Valid → Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")
        print(f"→ New best model saved (Acc: {best_val_acc:.4f})")


=== Epoch 1/12 ===


100%|██████████| 178/178 [00:20<00:00,  8.48it/s]


Train → Loss: 0.5490, Acc: 0.7981


100%|██████████| 63/63 [00:02<00:00, 24.20it/s]


Valid → Loss: 0.3808, Acc: 0.8344
→ New best model saved (Acc: 0.8344)

=== Epoch 2/12 ===


...

Valid → Loss: 0.3685, Acc: 0.9060

=== Epoch 12/12 ===


100%|██████████| 178/178 [00:20<00:00,  8.50it/s]


Train → Loss: 0.0094, Acc: 0.9991


100%|██████████| 63/63 [00:02<00:00, 24.20it/s]

Valid → Loss: 0.3705, Acc: 0.9060


In [213]:
# Load saved model
model.load_state_dict(torch.load("best_model.pt"))

# Evaluate on test set
test_loss, test_acc, test_preds, test_labels = eval_model(
    model=model,
    data_loader=test_loader,
    loss_fn=loss_fn,
    device=device
)

print(f"Test Accuracy: {test_acc:.4f}")

100%|██████████| 16/16 [00:00<00:00, 24.36it/s]

Test Accuracy: 0.9101


In [214]:
import pandas as pd
from sklearn.metrics import classification_report

report = classification_report(
    test_labels,
    test_preds,
    output_dict=True
)

df = pd.DataFrame(report).transpose()

# Format to 4 decimal places
df = df.round(4)

print(df)


              precision  recall  f1-score   support
0                0.9490  0.9588    0.9538   97.0000
1                0.8919  0.8919    0.8919   74.0000
2                0.7647  0.7222    0.7429   18.0000
accuracy         0.9101  0.9101    0.9101    0.9101
macro avg        0.8685  0.8576    0.8629  189.0000
weighted avg     0.9091  0.9101    0.9095  189.0000


In [179]:
output_model_file = '/content/drive/MyDrive/Honour/Result/pytorch_roberta_actionability.bin'
output_vocab_dir = '/content/drive/MyDrive/Honour/Result'
results_output_file = '/content/drive/MyDrive/Honour/Result/seed1.csv'

# Save the best model
# torch.save(model.state_dict(), output_model_file)
# print(f"Model saved to {output_model_file}")

# Save tokenizer files
# tokenizer.save_pretrained(output_vocab_dir)
# print(f"Tokenizer files saved to {output_vocab_dir}")

# Save predictions and results
results_df = pd.DataFrame({
    'Feedback': test_df['Feedback'].values,
    'True_Label': test_labels,
    'Predicted_Label': test_preds
})

results_df.to_csv(results_output_file, index=False)
print(f"Predictions saved to {results_output_file}")

Predictions saved to /content/drive/MyDrive/Honour/Result/seed1.csv


In [ ]:
# Load the saved model
model = RobertaClassifier(num_labels=3)
model.load_state_dict(torch.load('/content/drive/MyDrive/Honour/Result/pytorch_roberta_actionability.bin'))
model.to(device)

def actionability_label_to_string(label):
    label_map = {
        0: "Not Actionable",
        1: "Somewhat Actionable",
        2: "Highly Actionable"
    }

def Process_feedback(feedback_text):
    # Tokenize the input
    inputs = tokenizer(
        feedback_text,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='pt'
    ).to(device)

    # Get model prediction
    with torch.no_grad():
        outputs = model(inputs['input_ids'], inputs['attention_mask'])
        predicted_class = outputs.argmax().item()

    return {
        'feedback': feedback_text,
        'predicted_class': predicted_class,
        'class_label': actionability_label_to_string(predicted_class)
    }

example_feedbacks = [
    "Good job overall, keep it up!",
    "Consider adding more comments to explain your code",
    "On line 45, replace the for-loop with vectorized operations for better performance",
    "The structure could be improved",
    "The method validate_input() should check for null values before processing",
    "The introduction could be more engaging",
    "The code is hard to follow",
    "Add error handling for the file reading operation in load_data()"
]

results = [Process_feedback(fb) for fb in example_feedbacks]

for result in results:
    print(f"\nFeedback: {result['feedback']}")
    print(f"Predicted Class: {result['predicted_class']}")
    print("-"*50)

# Save results to JSON
import json
results_file = '/content/drive/MyDrive/Honour/Result/inference_results.json'
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {results_file}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Feedback: Good job overall, keep it up!
Predicted Class: 0
--------------------------------------------------

Feedback: Consider adding more comments to explain your code
Predicted Class: 1
--------------------------------------------------

Feedback: On line 45, replace the for-loop with vectorized operations for better performance
Predicted Class: 2
--------------------------------------------------

Feedback: The structure could be improved
Predicted Class: 1
--------------------------------------------------

Feedback: The method validate_input() should check for null values before processing
Predicted Class: 2
--------------------------------------------------

Feedback: The introduction could be more engaging
Predicted Class: 1
--------------------------------------------------

Feedback: The code is hard to follow
Predicted Class: 1
--------------------------------------------------

Feedback: Add error handling for the file reading operation in load_data()
Predicted Class: 2


In [ ]:
verbs_not_actionable = [
    "Your explanation links the early stages of the algorithm to real-world applications in a way that holds attention from start to finish",
    "The argument develops with a steady rhythm, carrying the reader through complex economic reasoning without breaking the narrative flow",
    "This description of the marine ecosystem captures intricate food-web relationships and keeps the audience engaged throughout the analysis",
    "The paper conveys intricate philosophical debates and maintains clarity even when the subject matter turns highly abstract",
    "Through careful pacing the historical overview guides readers across centuries while maintaining a clear sense of cause and consequence",
    "Your narrative integrates data and anecdote seamlessly, creating an account that feels both rigorous and lively",
    "The sequence of case studies builds momentum and provides a sense of discovery as each new example unfolds",
    "The discussion moves smoothly from statistical detail to broad implications, making dense material unexpectedly accessible",
    "A steady layering of evidence gives the section on renewable energy a compelling sense of inevitability",
    "The section on genetic editing maintains technical precision while sustaining an almost story-like quality",
    "Transitions between the different schools of literary theory flow naturally and keep the argument cohesive",
    "The depiction of market cycles carries an almost cinematic tension that keeps the reader curious about the next shift",
    "By combining concise definitions with vivid imagery the chapter makes abstract mathematical concepts easier to picture",
    "The comparison of rival marketing strategies unfolds like a narrative and keeps the strategic stakes clear",
    "Your account of classical architecture builds a picture of design evolution that remains engaging and well balanced",
    "The description of neural pathways provides a sense of movement and complexity without overwhelming the reader",
    "Each part of the economic model is introduced in a way that sustains interest and connects neatly to the previous section",
    "The report captures the atmosphere of early industrial innovation and sustains a sense of wonder about technological change",
    "Details of planetary motion are presented with a clarity that makes the scientific discoveries feel almost dramatic",
    "The analysis of political reforms carries the reader through shifting alliances and conveys the historical stakes vividly",
    "Scientific terminology is interwoven with accessible analogies so that the exposition feels natural and unforced",
    "The chronological arrangement of events builds a natural crescendo toward the final conclusion",
    "Your account of cultural interactions unfolds with precision and creates a strong sense of interconnectedness",
    "The mathematical reasoning proceeds with a calm logic that keeps even subtle steps easy to follow",
    "The exploration of competing ethical frameworks unfolds steadily and sustains a balanced, thoughtful tone"
]

verb_actionable = [
    "You should write clearer section headings in the literature review for easier navigation",
    "Add more detailed table captions at page two to explain statistical comparisons",
    "Provide an extra column in the spreadsheet to display cumulative monthly totals",
    "Make the contrast stronger between text and background in all presentation slides so it can be more clear",
    "Improve the essay by including a more explicit legend to clarify the biochemical pathway diagram",
    "The citation style should be aligned throughout the entire reference list, you have inconsistent style now",
    "You should add clearer labeling of anatomical structures in each biology diagram",
    "Provide standardized units in physics measurement table at page 32 for consistency",
    "Include distinct color palettes to differentiate data series in the dashboard",
    "Add descriptive footnotes to clarify economic model assumptions in the report for pages that is necessary",
    "Provide a dedicated column for customer feedback scores in the business report",
    "Ensure uniform spacing between paragraphs across the entire design critique document",
    "Add detailed captions for each marketing campaign image in the appendix",
    "Fix the number for tables and figures in the final thesis document, the number is not consistent at page 5, 7",
    "Insert an additional index page for quick access to key formulas at the beginning of your report ",
    "Use consistent bullet styles throughout the slides. For example, slide 2 uses number but slide 3 uses dot point",
    "Provide a complete scale bar on every microscopy image for reference",
    "Add an explicit title to each worksheet in the financial workbook",
    "Correct the alignment of all headers in the product specification sheet",
    "Include a separate appendix containing raw survey responses for transparency",
    "You need to explain more about the variables in the climate model graphs like add definitions",
    "Add a clear key to differentiate the categories in the financial risk heatmap",
    "You can improve the presentation by adding a slide include concise bullet points summarizing the main outcomes",
    "Add a dedicated section summarizing the main experimental limitations and list them as bullet points to improve clarity.",
    "Include a footer on every page that labels the dataset version used in the analysis so readers can easily track revisions."
]

# Label 0 for the first list, 2 for the second
feedback_examples = (
    [(txt, 0) for txt in verbs_not_actionable] +
    [(txt, 2) for txt in verb_actionable]
)


In [ ]:
results = []
for fb_text, true_label in feedback_examples:
    res = Process_feedback(fb_text)
    res['true_label'] = true_label
    results.append(res)

for r in results:
    print(
        f"\nFeedback: {r['feedback']}\n"
        f"Predicted Class: {r['predicted_class']} "
        f"True Label: {r['true_label']}\n"
        + "-"*60
    )


Feedback: Your explanation links the early stages of the algorithm to real-world applications in a way that holds attention from start to finish
Predicted Class: 0 True Label: 0
------------------------------------------------------------

Feedback: The argument develops with a steady rhythm, carrying the reader through complex economic reasoning without breaking the narrative flow
Predicted Class: 0 True Label: 0
------------------------------------------------------------

Feedback: This description of the marine ecosystem captures intricate food-web relationships and keeps the audience engaged throughout the analysis
Predicted Class: 0 True Label: 0
------------------------------------------------------------

Feedback: The paper conveys intricate philosophical debates and maintains clarity even when the subject matter turns highly abstract
Predicted Class: 0 True Label: 0
------------------------------------------------------------

Feedback: Through careful pacing the historical 

In [ ]:
from sklearn.metrics import accuracy_score

y_true = [r['true_label'] for r in results]
y_pred = [r['predicted_class'] for r in results]

acc = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {acc:.2%}\n")


Overall accuracy: 100.00%

